# Devoir NLP – Structuration du Code de la Route

##  Présentation

Nom : EL KORACHI INSAF  
Formation : Master Intelligence Artificielle et Sciences des Données (IASD)  
Établissement : Faculté des Sciences et Techniques de Tanger  

## Objectif du devoir

L’objectif de ce travail est de transformer un texte juridique arabe (Code de la route) en données structurées sous forme de fichier CSV.

Pour cela, nous avons utilisé des techniques de traitement automatique du langage naturel (NLP), notamment :
- le prétraitement du texte arabe,
- l’extraction d’informations par expressions régulières (Regex),
- la classification des articles,
- ainsi qu’une approche Machine Learning basée sur TF-IDF et KMeans.

In [7]:
import fitz
import re
import pandas as pd
import unicodedata

In [8]:
import fitz

pdf_path = r"C:\Users\LENOVO\Downloads\code de la route.pdf"

doc = fitz.open(pdf_path)

text = ""
for page in doc:
    text += page.get_text("text") + "\n"

print(text[:500])

2
ﻟﻠﺤﻜﻮﻣﺔ اﻟﻌﺎﻣﺔ اﻷﻣﺎﻧﺔ
 3 املادة
 يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها
 في املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تبديل
 .رخصتهم للسياقة تطبيقا للفقرات املوالية
 يمكن للمغاربة واألجانب الحاصلين على رخصة سياقة مسلمة من قبل دولة يربطها باملغرب اتفاق
 اعتراف متبادل بسندات السياقة، تبديل سنداتهم مقابل رخصة سياقة مغربية وفق الشروط املحددة
 .بمقتغضى االتفاق املذكور
 يمكن للحاصلين على رخصة سياقة مسلمة من قبل 


In [ ]:
import re
import pandas as pd
import unicodedata

def normalize_arabic(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)   # tashkeel
    text = re.sub(r'[إأآٱا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'[ؤئ]', 'ء', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ـ', '', text)
    text = re.sub(r'[^\S\r\n]+', ' ', text)
    text = re.sub(r'\n+', '\n', text).strip()
    return text
def extract_points(text):
    text = str(text)

    patterns = [
        r'خصم\s+(\d+)\s+نقط',
        r'سحب\s+(\d+)\s+نقط',
        r'فقدان\s+(\d+)\s+نقط'
    ]

    for p in patterns:
        match = re.search(p, text)
        if match:
            return match.group(1)

    return ""
def extract_fine(text):
    text = str(text)

    patterns = [
        r'(\d+)\s*(درهم|دراهم)',
        r'غرامة\s*(?:من)?\s*(\d+)',
        r'(\d+)\s*dh'
    ]

    for p in patterns:
        match = re.search(p, text)
        if match:
            return match.group(1)

    return ""
def extract_vehicle_category(text):
    text = str(text)

    # 1) catégories très spécifiques d'abord
    if any(w in text for w in ['عسكرية', 'العسكرية', 'عسكري']):
        return "militaire"

    if any(w in text for w in [
        'فلاحية', 'فلاحيه', 'غابوية', 'غابويه',
        'الأشغال العمومية', 'اشغال عموميه',
        'أشغال عمومية', 'اريبة', 'خاصة ذات محرك'
    ]):
        return "specialise"

    if any(w in text for w in ['دراجة', 'دراجه', 'نارية', 'النارية']):
        return "moto"

    if any(w in text for w in [
        'شاحنة', 'شاحنه', 'نقل البضائع',
        'حمولة', 'وزن', 'حافلة', 'حافله'
    ]):
        return "poids lourd"

    if any(w in text for w in ['سيارة', 'سياره']):
        return "léger"

    # 2) cas généraux : règle applicable sans type précis
    if any(w in text for w in [
        'رخصة السياقة', 'رخصه السياقه', 'رخصة دولية',
        'السير الدولي', 'السياقة على التراب الوطني',
        'صنف رخصة السياقة', 'أصناف رخصة السياقة',
        'مركبة ذات محرك', 'مجموعة مركبات'
    ]):
        return "general"

    return "general"
clean_text = normalize_arabic(text)

ARTICLE_PATTERN = re.compile(
    r'(?:(?:الماده|المادة|املاده|املادة)\s*(\d+)|(\d+)\s*(?:الماده|المادة|املاده|املادة))'
)

def split_articles(text):
    matches = list(ARTICLE_PATTERN.finditer(text))
    articles = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        article_text = text[start:end].strip()

        # ignorer articles trop courts
        if len(article_text) > 30:
            articles.append(article_text)

    # supprimer doublons exacts
    unique_articles = []
    seen = set()
    for a in articles:
        key = re.sub(r'\s+', ' ', a.strip())
        if key not in seen:
            seen.add(key)
            unique_articles.append(a)

    return unique_articles

articles = split_articles(clean_text)

def extract_article_id(article_text):
    m = ARTICLE_PATTERN.search(article_text)
    if not m:
        return None
    return m.group(1) or m.group(2)

def clean_noise(text):
    # supprimer références juridiques
    text = re.sub(r'ج\.ر.*', '', text)
    text = re.sub(r'رقم\s+\d+\.\d+.*', '', text)
    text = re.sub(r'من\s+\d+\s+تاريخ.*', '', text)

    # supprimer gros nombres
    text = re.sub(r'\b\d{3,}\b', '', text)

    # supprimer codes type J012
    text = re.sub(r'[A-Za-z]+\d+', '', text)

    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_infraction_desc(article_text):
    article_text = ARTICLE_PATTERN.sub('', article_text, count=1).strip()
    article_text = clean_noise(article_text)
    return article_text[:300]

def extract_fine(article_text):
    patterns = [
        r'(\d+)\s*(?:درهم|دراهم)',
        r'غرامه\s*(?:من)?\s*(\d+)'
    ]
    for p in patterns:
        m = re.search(p, article_text)
        if m:
            return m.group(1)
    return None

def extract_points(article_text):
    patterns = [
        r'خصم\s+(\d+)\s+نقط',
        r'سحب\s+(\d+)\s+نقط',
        r'فقدان\s+(\d+)\s+نقط'
    ]
    for p in patterns:
        m = re.search(p, article_text)
        if m:
            return m.group(1)
    return None

def classify_role(article_text):
    if any(w in article_text for w in ['يعاقب', 'غرامه', 'عقوبه', 'خصم', 'سحب']):
        return "sanction"
    elif any(w in article_text for w in ['يجب', 'يلزم', 'لا يجوز']):
        return "obligation"
    elif any(w in article_text for w in ['يراد', 'يقصد', 'تعريف']):
        return "definition"
    elif 'يجوز' in article_text:
        return "autorisation"
    else:
        return "autre"

keywords_list = [
    'سرعه', 'سياقه', 'رخصه', 'مركبه', 'طريق',
    'نقط', 'غرامه', 'نقل', 'اشخاص',
    'تسجيل', 'دولي', 'عسكري'
]

def extract_keywords(article_text):
    found = [kw for kw in keywords_list if kw in article_text]
    return ", ".join(found)

rows = []
seen_ids = set()

for article in articles:
    article_id = extract_article_id(article)

    if article_id is not None and article_id not in seen_ids:
        seen_ids.add(article_id)

        rows.append({
            "article_id": article_id,
            "infraction_desc": extract_infraction_desc(article),
            "categorie_vehicule": extract_vehicle_category(article),
            "amende_fixe": extract_fine(article),
            "points_retrait": extract_points(article),
            "mots_cles": extract_keywords(article),
            "type_paragraphe": classify_role(article)
        })
print("Nombre de lignes extraites :", len(rows))
print(rows[:3])

Nombre de lignes extraites : 6
[{'article_id': '3', 'infraction_desc': 'يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها في املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه املغربيه، او ان يطلبوا تبديل .رخصتهم للسياقه تطبيقا للفقرات املواليه يمكن للمغاربه واالجانب الحاصلين علي رخصه سياقه مسلمه من قبل دوله يربطها باملغرب ا', 'categorie_vehicule': 'general', 'amende_fixe': None, 'points_retrait': None, 'mots_cles': 'سياقه, رخصه', 'type_paragraphe': 'obligation'}, {'article_id': '4', 'infraction_desc': 'في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهيءات املءهله لذلك من .قبل االداره، رخصه دوليه للسياقه موضوعه في دفتر خاص يجوز للساءقين من جنسيه اجنبيه، الحاصلين علي رخصه دوليه للسياقه، السياقه علي التراب 2 من', 'categorie_vehicule': 'general', 'amende_fixe': None, 'points_retrait': None, 'mots_cles': 'سياقه, رخصه, دولي', 'type_paragraphe': 'autorisation'}, {'article_id': '2', 'infraction_desc': 'الوطني خالل مده صالحيه ا

In [10]:
test = "يعاقب بغرامة 300 درهم مع خصم 2 نقط"

print(extract_fine(test))    # 300
print(extract_points(test))  # 2

300
2


## Validation de l’extraction

Un test a été réalisé pour vérifier le bon fonctionnement des fonctions d’extraction.

Exemple :
"يعاقب بغرامة 300 درهم مع خصم 2 نقط"

Résultat :
- amende_fixe = 300
- points_retrait = 2

Cela confirme que les fonctions implémentées permettent d’extraire correctement ces informations lorsqu’elles sont présentes dans le texte.

In [11]:
import os

df = pd.DataFrame(rows)
df = df.sort_values("article_id").reset_index(drop=True)
df = df.fillna("")

os.makedirs("output", exist_ok=True)
df.to_csv("output/export_final.csv", index=False, encoding="utf-8-sig")
print("Fichier généré : output/export_final.csv")
df.head(10)

Fichier généré : output/export_final.csv


,article_id,infraction_desc,categorie_vehicule,amende_fixe,points_retrait,mots_cles,type_paragraphe
0,2,الوطني خالل مده صالحيه الرخصه املذكوره دون ان ...,general,,,رخصه,autre
1,3,يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه...,general,,,"سياقه, رخصه",obligation
2,4,في حاله السير الدولي ووفقا لالتفاقيه الدوليه ل...,general,,,"سياقه, رخصه, دولي",autorisation
3,5,من 13 بتاريخ 1.16. رقم الشريف 012ال 34567بت ر8...,militaire,,,"سياقه, رخصه, طريق, عسكري",autorisation
4,6,ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او...,specialise,,,"سياقه, رخصه, مركبه, طريق",autorisation
5,7,من 13 بتاريخ 1.16. رقم الشريف 012ال 34567بت ر8...,moto,,,"سياقه, رخصه, نقل",autre


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# textes
texts = df["infraction_desc"].fillna("").astype(str)

# garder textes non vides
valid_idx = texts[texts.str.strip() != ""].index
texts_valid = texts.loc[valid_idx]

# TF-IDF
vectorizer = TfidfVectorizer(max_features=50)
X = vectorizer.fit_transform(texts_valid)

# nombre de clusters adapté
n_clusters = min(2, X.shape[0])

if n_clusters >= 2:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df.loc[valid_idx, "cluster"] = kmeans.fit_predict(X)
else:
    df["cluster"] = 0

df.head(10)

,article_id,infraction_desc,categorie_vehicule,amende_fixe,points_retrait,mots_cles,type_paragraphe,cluster
0,2,الوطني خالل مده صالحيه الرخصه املذكوره دون ان ...,general,,,رخصه,autre,0.0
1,3,يجب علي الساءقين الحاصلين علي رخصه سياقه مسلمه...,general,,,"سياقه, رخصه",obligation,0.0
2,4,في حاله السير الدولي ووفقا لالتفاقيه الدوليه ل...,general,,,"سياقه, رخصه, دولي",autorisation,0.0
3,5,من 13 بتاريخ 1.16. رقم الشريف 012ال 34567بت ر8...,militaire,,,"سياقه, رخصه, طريق, عسكري",autorisation,1.0
4,6,ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او...,specialise,,,"سياقه, رخصه, مركبه, طريق",autorisation,0.0
5,7,من 13 بتاريخ 1.16. رقم الشريف 012ال 34567بت ر8...,moto,,,"سياقه, رخصه, نقل",autre,1.0


Une approche par règles (Regex) et une approche ML (TF-IDF + KMeans) ont été utilisées pour structurer et analyser les articles du code de la route.


## Analyse des clusters

Nous avons utilisé l’algorithme KMeans pour regrouper automatiquement les articles en fonction de leur similarité.

Après vectorisation des textes avec TF-IDF, chaque article est représenté sous forme numérique, ce qui permet de mesurer la similarité entre les textes.

Les résultats montrent que :

- Le cluster 0 regroupe les articles ayant un contenu général lié aux règles de conduite et aux permis.
- Le cluster 1 regroupe des articles plus spécifiques, notamment ceux liés à des cas particuliers (ex : véhicules militaires ou spécialisés).

Ainsi, le clustering permet d’identifier automatiquement des groupes d’articles similaires sans intervention manuelle.